In [1]:
from pyspark.sql import functions as F
df_gold = spark.read.table("gold_flipkart_ai_products")


gold_pricing = df_gold.groupBy("main_category", "price_category").agg(
    F.count("*").alias("product_count"),
    F.round(F.avg("retail_price"), 0).alias("avg_retail_price"),
    F.round(F.avg("discounted_price"), 0).alias("avg_discounted_price"),
    F.round(F.avg("discount_percentage"), 1).alias("avg_discount_pct"),
    F.round(F.avg("ai_quality_score"), 1).alias("avg_ai_quality")
).orderBy(F.desc("product_count"))


gold_pricing.write.format("delta").mode("overwrite").saveAsTable("gold_pricing_by_category")


gold_customer = df_gold.groupBy("ai_target_customer", "main_category").agg(
    F.count("*").alias("product_count"),
    F.round(F.avg("discounted_price"), 0).alias("avg_price"),
    F.round(F.avg("discount_percentage"), 1).alias("avg_discount")
).orderBy(F.desc("product_count"))

gold_customer.write.format("delta").mode("overwrite").saveAsTable("gold_customer_segments")


gold_quality = df_gold.groupBy("ai_quality_score", "price_category").agg(
    F.count("*").alias("product_count"),
    F.round(F.avg("discount_percentage"), 1).alias("avg_discount")
).orderBy(F.desc("ai_quality_score"))
gold_quality.write.format("delta").mode("overwrite").saveAsTable("gold_quality_distribution")
gold_top_discounts = df_gold.select(
    "product_name", "main_category", "brand",
    "retail_price", "discounted_price", "discount_percentage",
    "ai_quality_score", "ai_target_customer", "ai_key_features"
).orderBy(F.desc("discount_percentage")).limit(100)

gold_top_discounts.write.format("delta").mode("overwrite").saveAsTable("gold_top_discounts")
print("Created gold_pricing_by_category, gold_customer_segments, gold_quality_distribution, gold_top_discounts")


StatementMeta(, 1557cb70-1157-4b6e-82b4-a5f04ec76024, 3, Finished, Available, Finished, False)

Created gold_pricing_by_category, gold_customer_segments, gold_quality_distribution, gold_top_discounts
